# ADNI MCI Stability Classifier — Full Training & Saving Pipeline
## Two Models Saved Per Stream:
### 1. RNN cfg2 (hidden=128, layers=3) + By-Domain PCA  |  2. LightGBM on Trajectory Summary Features

**Streams:** Full feature set & Algerian feature set  
**Target:** 3-class stability label — `CN` / `MCI_stable` / `MCI_converting`

---
### Key design facts
| Model | Input representation | Best F1 (full) | Best F1 (alg) |
|---|---|---|---|
| **RNN cfg2 + domain PCA** | Padded visit sequence (T × 36 PCA dims) | 0.8157 | 0.8187 |
| **LightGBM** | One row per patient — trajectory summary features (slope, AUC, decline…) | 0.7922 | 0.7922 |

The two models use **completely different data pipelines**:
- The RNN sees the raw longitudinal sequence reduced with by-domain PCA
- LightGBM receives one summarised row per patient (no sequence, no PCA)

Both pipelines are fully self-contained for inference.


## 0. Imports & Config

In [1]:
import warnings, random, time, copy, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from scipy.stats import linregress
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, classification_report)
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── RNN cfg2: CORRECT config from dim-red experiments ────────────────
RNN_CFG2 = {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.3,
             'lr': 3e-4, 'batch_size': 16}

# ── Training budget ───────────────────────────────────────────────────
RNN_EPOCHS   = 100
RNN_PATIENCE = 12
N_PER_DOMAIN = 6        # PCA components kept per clinical domain
LGB_TRIALS   = 40       # Optuna trials for LightGBM HPO

SAVE_ROOT = './stability_models'
os.makedirs(f'{SAVE_ROOT}/full', exist_ok=True)
os.makedirs(f'{SAVE_ROOT}/alg',  exist_ok=True)
print(f'Models will be saved to: {SAVE_ROOT}/')


Device: cpu
Models will be saved to: ./stability_models/


## 1. Data Paths — Update These

In [2]:
DATA_PATH = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/adni_stability.csv'
MTA_PATH  = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/MTA_labels_final.csv'
AMY_PATH  = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/amy_dataset.csv'


## 2. Feature Definitions

In [3]:
TARGET_COL = 'STABILITY_LABEL'

LEAKING_COLS = ['DIAGNOSIS', 'DIAGNOSIS_LABEL', 'TRAJECTORY_CLEANED',
                'DXMDUE', 'DXDEP', 'DXCONFID', 'OBS_SPAN_YEARS', 'DXDSEV']

COLS_ALG = [
    "RID", "VISCODE2", "PTGENDER", "age", "PTHAND", "PTMARRY", "PTEDUCAT", "PTWORK", "PTNOTRT",
    "VISDATE",
    "MMDATE","MMYEAR","MMMONTH","MMDAY","MMREAD","MMWRITE","MMDRAW","MMREPEAT","MMSEASON",
    "MMHOSPIT","MMFLOOR","MMCITY","WORD1","WORD2","WORD3","MMSCORE","MOCA","CUBE",
    "CLOCKCON","CLOCKNO","CLOCKHAN","DIGFOR","DIGBACK",
    "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5","REPEAT1","REPEAT2","FFLUENCY",
    "FAQFORM","FAQFINAN","FAQSHOP","FAQGAME","FAQBEVG","FAQMEAL","FAQEVENT","FAQTV","FAQREM","FAQTRAVL","FAQ",
    "Creatinine","Calcium","Direct Bilirubin","Platelet Ct.","Red Blood Cell Count",
    "Thyroid-stimulating hormone","Total Bilirubin","glucose","Vitamin B12","White Blood Cell Count",
    "Abeta42","Abeta40","Abeta_ratio","Homocysteine","Hemoglobin A1C",
    "Tau181","pTau181","Gamma-Glutamyltransferase","Hematocrit","Hemoglobin",
    "MOTHDEM","MOTHAD","MOTHSXAGE","FATHDEM","FATHAD","FATHSXAGE",
    "SIBGENDER","SIBDEMENT","SIBAD","SIBSXAGE",
    "IHSYMPTOM","IHDESC","IHCHRON","IHSEVER","IHPRESENT","IHSURG",
    "MH19OTHR","MHPSYCH","MH2NEURL","MH3HEAD","MH4CARD","MH5RESP","MH6HEPAT","MH7DERM",
    "MH8MUSCL","MH9ENDO","MH14BALCH","MH14CALCH","MH10GAST","MH11HEMA","MH12RENA",
    "MH13ALLE","MH14ALCH","MH14AALCH","MH17MALI","MH18SURG","MH15DRUG","MH15ADRUG","MH15BDRUG",
    "MH16SMOK","MH16ASMOK","MH16BSMOK","MH16CSMOK",
    "BSXSYMNO","BSXSEVER","BSXCHRON","KEYMED","CMMED","CMDOSE","CMREASON",
    'AMYLOID_STATUS','MTA_ATROPHY', TARGET_COL,
]

# Columns for which trajectory features are computed (slope, AUC, decline…)
TRAJECTORY_COLS = [
    "MMSCORE","MOCA","FFLUENCY","DIGFOR","DIGBACK","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
    "FAQ","FAQFORM","FAQFINAN","FAQMEAL","FAQEVENT",
    "Abeta42","Abeta40","Abeta_ratio","Tau181","pTau181",
    "AMYLOID_STATUS","MTA_ATROPHY",
]

# Time-invariant patient features used as static branch in RNN
STATIC_FEATURES = [
    "PTGENDER","age","PTHAND","PTMARRY","PTEDUCAT","PTWORK","PTNOTRT",
    "MOTHDEM","MOTHAD","FATHDEM","FATHAD","SIBGENDER","SIBDEMENT","SIBAD",
    "MH2NEURL","MH4CARD","MH9ENDO","MH16SMOK",
]

# Domain groups for by-domain PCA
DOMAIN_GROUPS = {
    'cognitive':  ["MMDATE","MMYEAR","MMMONTH","MMDAY","MMREAD","MMWRITE","MMDRAW","MMREPEAT",
                   "MMSEASON","MMHOSPIT","MMFLOOR","MMCITY","WORD1","WORD2","WORD3","MMSCORE",
                   "MOCA","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN","DIGFOR","DIGBACK",
                   "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5","REPEAT1","REPEAT2","FFLUENCY"],
    'functional': ["FAQFORM","FAQFINAN","FAQSHOP","FAQGAME","FAQBEVG","FAQMEAL",
                   "FAQEVENT","FAQTV","FAQREM","FAQTRAVL","FAQ"],
    'biomarker':  ["Abeta42","Abeta40","Abeta_ratio","Tau181","pTau181","Homocysteine",
                   "Hemoglobin A1C","Creatinine","Calcium","Direct Bilirubin","Total Bilirubin",
                   "Platelet Ct.","Red Blood Cell Count","Thyroid-stimulating hormone",
                   "glucose","Vitamin B12","White Blood Cell Count",
                   "Gamma-Glutamyltransferase","Hematocrit","Hemoglobin",
                   "AMYLOID_STATUS","MTA_ATROPHY"],
    'demographics': ["PTGENDER","age","PTHAND","PTMARRY","PTEDUCAT","PTWORK","PTNOTRT"],
    'medical_history': ["MOTHDEM","MOTHAD","MOTHSXAGE","FATHDEM","FATHAD","FATHSXAGE",
                        "SIBGENDER","SIBDEMENT","SIBAD","SIBSXAGE",
                        "IHSYMPTOM","IHDESC","IHCHRON","IHSEVER","IHPRESENT","IHSURG",
                        "MH2NEURL","MH3HEAD","MH4CARD","MH5RESP","MH6HEPAT","MH7DERM",
                        "MH8MUSCL","MH9ENDO","MH10GAST","MH11HEMA","MH12RENA","MH13ALLE",
                        "MH14ALCH","MH15DRUG","MH16SMOK","MH17MALI","MH18SURG","MHPSYCH",
                        "BSXSYMNO","BSXSEVER","BSXCHRON"],
}

LABEL_MAP_3 = {
    'CN': 'CN', 'MCI stable': 'MCI_stable',
    'MCI unstable': 'MCI_converting', 'Dementia': 'MCI_converting', 'MCI': None,
}
print('Feature definitions ready.')


Feature definitions ready.


## 3. Load & Merge Data

In [4]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
mta = pd.read_csv(MTA_PATH)
amy = pd.read_csv(AMY_PATH)
raw = raw.merge(mta[['RID','VISCODE2','MTA_ATROPHY']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')
raw = raw.merge(amy[['RID','VISCODE2','AMYLOID_STATUS']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')
print(f'Raw shape: {raw.shape} | Patients: {raw["RID"].nunique()}')
print(raw[TARGET_COL].value_counts())


Raw shape: (22071, 148) | Patients: 4798
STABILITY_LABEL
CN              6583
MCI stable      5657
MCI unstable    4589
Dementia        2293
MCI             1926
Name: count, dtype: int64


## 4. Preprocessing Pipeline

Shared by both models. Produces:
- `df`: visit-level DataFrame with delta / rate / baseline-relative features (used by RNN)
- `feature_cols`, `static_cols`, `le_target`, `cat_le_map`, `global_means` (saved for inference)


In [5]:
def viscode_to_month(vc):
    if pd.isna(vc): return np.nan
    vc = str(vc).strip().lower()
    if vc in ('sc','f','uns'): return -1
    if vc in ('bl','v01'):     return 0
    if vc.startswith('m'):
        try: return int(vc[1:])
        except: return np.nan
    return np.nan


def preprocess(df_raw, feature_mode='full', label_mode='3class'):
    """
    Full preprocessing pipeline — identical to the best-performing version in
    stability-classifier-last-try-for-real.ipynb.

    Returns: df, feature_cols, static_cols, le_target, cat_le_map, global_means
    cat_le_map and global_means are saved for inference.
    """
    df = df_raw.copy()

    # 3-class label remapping
    if label_mode == '3class':
        df[TARGET_COL] = df[TARGET_COL].map(LABEL_MAP_3)
        n_before = df['RID'].nunique()
        df = df[df[TARGET_COL].notna()]
        print(f'  Dropped {n_before - df["RID"].nunique()} patients with undetermined MCI')

    # Month encoding & sort
    df['_month'] = df['VISCODE2'].apply(viscode_to_month)
    df = df.sort_values(['RID','_month'])

    # Dementia trim for converting patients
    def trim_dementia(grp):
        lbl = grp[TARGET_COL].iloc[0]
        if lbl in ('MCI unstable','MCI_converting'):
            dem_mask = grp['DIAGNOSIS_LABEL'] == 'Dementia'
            if dem_mask.any():
                grp = grp.iloc[:dem_mask.values.argmax()]
        return grp
    df = df.groupby('RID', group_keys=False).apply(trim_dementia)
    df = df[df['RID'].isin(df.groupby('RID').size()[lambda x: x >= 1].index)]
    print(f'  After dementia trim: {df.shape}')

    # Feature selection
    if feature_mode == 'algerian':
        keep = [c for c in COLS_ALG if c in df.columns]
        df   = df[list(set(keep + ['RID','_month','DIAGNOSIS_LABEL']))]
    else:
        drop = [c for c in LEAKING_COLS if c in df.columns and c != TARGET_COL]
        df   = df.drop(columns=drop, errors='ignore')

    # VISCODE2 / VISDATE → numeric
    if 'VISCODE2' in df.columns:
        df['VISCODE2_num'] = df['_month']; df = df.drop(columns=['VISCODE2'], errors='ignore')
    if 'VISDATE' in df.columns:
        df['VISDATE'] = pd.to_datetime(df['VISDATE'], errors='coerce')
        df['VISDATE_days'] = df.groupby('RID')['VISDATE'].transform(lambda x: (x - x.min()).dt.days)
        df = df.drop(columns=['VISDATE'], errors='ignore')

    # Categorical encoding — store encoders for inference
    ignore = {'RID', TARGET_COL, '_month', 'DIAGNOSIS_LABEL'}
    cat_cols = [c for c in df.select_dtypes(include=['object','string']).columns if c not in ignore]
    cat_le_map = {}
    for c in cat_cols:
        mode_val = df[c].mode(dropna=True)
        mode_val = mode_val.iloc[0] if len(mode_val) else 'UNKNOWN'
        df[c] = df[c].fillna(mode_val)
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))
        cat_le_map[c] = le

    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ignore]

    # Per-patient mean imputation → global fallback
    df[num_cols] = df.groupby('RID')[num_cols].transform(lambda x: x.fillna(x.mean()))
    global_means = df[num_cols].mean()
    df[num_cols] = df[num_cols].fillna(global_means)

    # Missing-indicator flags (only for truly missing columns)
    flag_cols = []
    for c in num_cols:
        if c in df_raw.columns and df_raw.loc[df.index, c].isnull().mean() > 0.005:
            df[f'{c}_missing'] = df_raw.loc[df.index, c].isnull().astype(int).values
            flag_cols.append(f'{c}_missing')

    # Delta features (single-step change)
    traj_present = [c for c in TRAJECTORY_COLS if c in df.columns]
    for c in traj_present:
        df[f'{c}_delta'] = df.groupby('RID')[c].diff().fillna(0)

    # Rate-of-change (delta / elapsed months) — removes visit-gap confound
    if '_month' in df.columns:
        df['_month_gap'] = df.groupby('RID')['_month'].diff().fillna(1).clip(lower=1)
        for c in traj_present:
            df[f'{c}_rate'] = df[f'{c}_delta'] / df['_month_gap']
        df = df.drop(columns=['_month_gap'], errors='ignore')

    # Baseline-relative features
    for c in traj_present:
        bl = df.groupby('RID')[c].transform('first')
        df[f'{c}_from_bl'] = df[c] - bl

    # Final feature list
    skip = ignore | {'_month','TRAJECTORY_CLEANED','OBS_SPAN_YEARS','DIAGNOSIS_LABEL','DIAGNOSIS'}
    feature_cols = [c for c in df.columns
                    if c not in skip and c in df.select_dtypes(include=[np.number]).columns]
    static_cols  = [c for c in STATIC_FEATURES if c in feature_cols]

    le_target = LabelEncoder()
    df['target'] = le_target.fit_transform(df[TARGET_COL].astype(str))

    print(f'  Features: {len(feature_cols)} | Static: {len(static_cols)} | Classes: {list(le_target.classes_)}')
    return df, feature_cols, static_cols, le_target, cat_le_map, global_means[num_cols]


print('=== FULL ===')
df_full, feat_full, static_full, le_full, cat_le_full, gmeans_full = preprocess(raw, 'full')

print('\n=== ALGERIAN ===')
df_alg, feat_alg, static_alg, le_alg, cat_le_alg, gmeans_alg = preprocess(raw, 'algerian')


=== FULL ===
  Dropped 1815 patients with undetermined MCI
  After dementia trim: (14934, 149)
  Features: 321 | Static: 18 | Classes: ['CN', 'MCI_converting', 'MCI_stable']

=== ALGERIAN ===
  Dropped 1815 patients with undetermined MCI
  After dementia trim: (14934, 149)
  Features: 291 | Static: 18 | Classes: ['CN', 'MCI_converting', 'MCI_stable']


## 5. Patient-Level Split
Splits by RID (80/10/10). Produces both sequence splits (for RNN) and flat splits (for LightGBM).

In [6]:
def patient_split(df, feature_cols, seed=SEED, train_frac=0.8, val_frac=0.1):
    """
    Returns:
      seq_*  : list of (X_tensor [T,F], y_int, rid) — for RNN
      flat_* : DataFrame slice by RID — for trajectory summary / LightGBM
      meta   : dict with mu, std, train/val/test RID sets
    """
    rids = df['RID'].unique()
    rng  = np.random.default_rng(seed); rng.shuffle(rids)
    n    = len(rids)
    n_tr = int(n * train_frac); n_va = int(n * val_frac)
    train_rids = set(rids[:n_tr])
    val_rids   = set(rids[n_tr:n_tr+n_va])
    test_rids  = set(rids[n_tr+n_va:])
    print(f'  Train={len(train_rids)} | Val={len(val_rids)} | Test={len(test_rids)} patients')

    tr_mask = df['RID'].isin(train_rids)
    mu  = df.loc[tr_mask, feature_cols].mean().values.astype(np.float32)
    std = df.loc[tr_mask, feature_cols].std().replace(0, 1).values.astype(np.float32)

    def build_seqs(rid_set):
        seqs = []
        for rid, grp in df[df['RID'].isin(rid_set)].groupby('RID'):
            g = grp.sort_values('_month') if '_month' in grp.columns else grp
            X = np.nan_to_num((g[feature_cols].values.astype(np.float32) - mu) / std, nan=0.)
            y = int(g['target'].iloc[-1])
            seqs.append((torch.tensor(X), y, rid))
        return seqs

    meta = dict(mu=mu, std=std, train_rids=train_rids, val_rids=val_rids, test_rids=test_rids)
    return (build_seqs(train_rids), build_seqs(val_rids), build_seqs(test_rids),
            df[df['RID'].isin(train_rids)], df[df['RID'].isin(val_rids)],
            df[df['RID'].isin(test_rids)], meta)


print('=== FULL split ===')
seq_tr_f, seq_va_f, seq_te_f, flat_tr_f, flat_va_f, flat_te_f, meta_f = patient_split(df_full, feat_full)

print('\n=== ALG split ===')
seq_tr_a, seq_va_a, seq_te_a, flat_tr_a, flat_va_a, flat_te_a, meta_a = patient_split(df_alg, feat_alg)


=== FULL split ===
  Train=1932 | Val=241 | Test=243 patients

=== ALG split ===
  Train=1932 | Val=241 | Test=243 patients


## 6. Trajectory Summary Features (LightGBM Input)

For each patient, each trajectory column is aggregated into: slope, AUC, first, last, min, max, std, cumulative_decline. Plus means of derived delta/rate/baseline columns. This gives one flat row per patient.

In [7]:
def _trapz(y, x):
    """Trapezoidal integration compatible with all numpy versions."""
    try:
        return np.trapz(y, x)
    except AttributeError:
        return float(np.sum((np.diff(x)) * (y[:-1] + y[1:]) / 2.0))


def compute_trajectory_summary(df, feature_cols):
    """
    Aggregate each patient's visit sequence → one summary row.
    This is the exact data representation fed to LightGBM.

    For each TRAJECTORY_COL column:
      slope         — linear regression over month index (direction + speed)
      auc           — trapezoidal area under the visit curve / span
      first/last    — absolute anchors
      min/max/std   — range and variability
      cum_decline   — sum of negative step-changes only (asymmetric decline signal)

    Also averages delta/rate/from_bl columns (computed in preprocessing).
    Static patient features are taken from the first visit.
    """
    traj_present = [c for c in TRAJECTORY_COLS if c in df.columns]
    extra_summ   = [c for c in feature_cols
                    if c.endswith('_delta') or c.endswith('_rate') or c.endswith('_from_bl')]
    rows = []
    for rid, grp in df.groupby('RID'):
        g      = grp.sort_values('_month') if '_month' in grp.columns else grp
        months = g['_month'].values.astype(float)
        row    = {'RID': rid, 'target': int(g['target'].iloc[-1])}

        # Visit meta
        row['n_visits']          = len(g)
        row['total_span_months'] = months[-1] - months[0] if len(months) > 1 else 0
        row['mean_visit_gap']    = row['total_span_months'] / max(len(g)-1, 1)

        for c in traj_present:
            if c not in g.columns:
                for sfx in ('_slope','_auc','_first','_last','_min','_max','_std','_cum_decline'):
                    row[f'{c}{sfx}'] = 0.0
                continue
            vals = g[c].values.astype(float)
            mask = ~np.isnan(vals)
            if mask.sum() < 1:
                for sfx in ('_slope','_auc','_first','_last','_min','_max','_std','_cum_decline'):
                    row[f'{c}{sfx}'] = 0.0
                continue
            v = vals[mask]; m = months[mask]
            slope = linregress(m, v).slope if len(v) >= 2 else 0.0
            span  = m[-1] - m[0]
            row[f'{c}_slope']       = slope
            row[f'{c}_auc']         = _trapz(v, m) / max(span, 1)
            row[f'{c}_first']       = v[0]
            row[f'{c}_last']        = v[-1]
            row[f'{c}_min']         = v.min()
            row[f'{c}_max']         = v.max()
            row[f'{c}_std']         = v.std() if len(v) > 1 else 0.0
            diffs = np.diff(v)
            row[f'{c}_cum_decline'] = diffs[diffs < 0].sum() if len(diffs) > 0 else 0.0

        for c in extra_summ:
            if c in g.columns:
                row[f'{c}_mean'] = g[c].mean()
                row[f'{c}_std']  = g[c].std() if len(g) > 1 else 0.0

        for c in STATIC_FEATURES:
            if c in g.columns:
                row[c] = g[c].iloc[0]

        rows.append(row)

    summary = pd.DataFrame(rows).fillna(0)
    feat_cols = [c for c in summary.columns if c not in ('RID','target')]
    print(f'  Summary: {len(summary)} patients × {len(feat_cols)} features')
    return summary, feat_cols


def split_summary(summ, meta):
    tr = summ[summ['RID'].isin(meta['train_rids'])]
    va = summ[summ['RID'].isin(meta['val_rids'])]
    te = summ[summ['RID'].isin(meta['test_rids'])]
    return tr, va, te


print('=== Trajectory summary — FULL ===')
summ_full, summ_feat_full = compute_trajectory_summary(df_full, feat_full)
str_tr_f, str_va_f, str_te_f = split_summary(summ_full, meta_f)

print('\n=== Trajectory summary — ALGERIAN ===')
summ_alg,  summ_feat_alg  = compute_trajectory_summary(df_alg,  feat_alg)
str_tr_a, str_va_a, str_te_a = split_summary(summ_alg,  meta_a)


=== Trajectory summary — FULL ===
  Summary: 2416 patients × 315 features

=== Trajectory summary — ALGERIAN ===
  Summary: 2416 patients × 315 features


## 7. LightGBM — Optuna HPO (Trajectory Summary)

Optuna TPE sampler over 40 trials. Best config retrained on train+val before final test scoring.

In [8]:
def lgb_objective(trial, X_tr, y_tr, X_va, y_va, n_classes):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'objective':'multiclass', 'num_class':n_classes,
        'metric':'multi_logloss', 'random_state':SEED, 'n_jobs':-1, 'verbosity':-1,
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    return f1_score(y_va, m.predict(X_va), average='macro', zero_division=0)


def train_lightgbm(summ_tr, summ_va, summ_te, feat_cols, le_target, n_trials=LGB_TRIALS, label='lgb'):
    """
    Run Optuna HPO for LightGBM on trajectory summary features.
    Returns the best model retrained on train+val.
    """
    X_tr = summ_tr[feat_cols].values.astype(np.float32)
    X_va = summ_va[feat_cols].values.astype(np.float32)
    X_te = summ_te[feat_cols].values.astype(np.float32)
    y_tr = summ_tr['target'].values
    y_va = summ_va['target'].values
    y_te = summ_te['target'].values
    nc   = len(le_target.classes_)

    print(f'  [{label}] Optuna LightGBM HPO ({n_trials} trials)...')
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t: lgb_objective(t, X_tr, y_tr, X_va, y_va, nc),
                   n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params
    best_params.update({'objective':'multiclass','num_class':nc,
                        'random_state':SEED,'verbosity':-1,'n_jobs':-1})
    print(f'    Best val F1: {study.best_value:.4f}')
    print(f'    Best params: {best_params}')

    # Retrain on train + val combined
    X_trva = np.vstack([X_tr, X_va])
    y_trva = np.concatenate([y_tr, y_va])
    model  = lgb.LGBMClassifier(**best_params)
    model.fit(X_trva, y_trva)

    preds   = model.predict(X_te)
    probs   = model.predict_proba(X_te)
    classes = list(le_target.classes_)
    f1_mac  = f1_score(y_te, preds, average='macro',    zero_division=0)
    f1_wt   = f1_score(y_te, preds, average='weighted', zero_division=0)
    try:    auc = roc_auc_score(y_te, probs, multi_class='ovr', average='macro')
    except: auc = float('nan')
    print(f'    Test F1_macro={f1_mac:.4f}  AUC={auc:.4f}')
    print(classification_report(y_te, preds, target_names=classes, zero_division=0))

    return {'model': model, 'best_params': best_params, 'study': study,
            'feat_cols': feat_cols, 'le_target': le_target,
            'f1_macro': f1_mac, 'f1_weighted': f1_wt, 'auc': auc,
            'preds': preds, 'labels': y_te, 'probs': probs, 'classes': classes}


print('=== LightGBM — FULL ===')
lgb_full = train_lightgbm(str_tr_f, str_va_f, str_te_f, summ_feat_full, le_full, label='full')

print('\n=== LightGBM — ALGERIAN ===')
lgb_alg  = train_lightgbm(str_tr_a, str_va_a, str_te_a, summ_feat_alg, le_alg, label='alg')


=== LightGBM — FULL ===
  [full] Optuna LightGBM HPO (40 trials)...
    Best val F1: 0.8335
    Best params: {'n_estimators': 554, 'num_leaves': 35, 'learning_rate': 0.039542590989192355, 'subsample': 0.5514679674687517, 'colsample_bytree': 0.8357187259686519, 'min_child_samples': 33, 'reg_alpha': 3.8372343456060154, 'reg_lambda': 9.83149493594911, 'objective': 'multiclass', 'num_class': 3, 'random_state': 42, 'verbosity': -1, 'n_jobs': -1}
    Test F1_macro=0.7922  AUC=0.9587
                precision    recall  f1-score   support

            CN       0.90      0.94      0.92       144
MCI_converting       0.86      0.73      0.79        52
    MCI_stable       0.65      0.68      0.67        47

      accuracy                           0.84       243
     macro avg       0.81      0.78      0.79       243
  weighted avg       0.84      0.84      0.84       243


=== LightGBM — ALGERIAN ===
  [alg] Optuna LightGBM HPO (40 trials)...
    Best val F1: 0.8335
    Best params: {'n_estima

## 8. By-Domain PCA (RNN Input)

Fits PCA independently per clinical domain (6 components each), then concatenates. This gives 36D input for the RNN vs 300D raw, dramatically reducing overfitting.

In [9]:
def get_domain_indices(feat_cols):
    col_idx  = {c: i for i, c in enumerate(feat_cols)}
    domain_idx = {}; assigned = set()
    for domain, keywords in DOMAIN_GROUPS.items():
        idxs = [col_idx[c] for c in feat_cols
                if (c in keywords or any(c.startswith(kw+'_') for kw in keywords))
                and c not in assigned]
        for c in feat_cols:
            if any(c == kw or c.startswith(kw+'_') for kw in keywords) and c not in assigned:
                if c in col_idx:
                    idxs.append(col_idx[c]); assigned.add(c)
        # deduplicate
        seen = set(); idxs = [x for x in idxs if not (x in seen or seen.add(x))]
        if idxs: domain_idx[domain] = idxs
    other = [col_idx[c] for c in feat_cols if c not in assigned]
    if other: domain_idx['other'] = other
    return domain_idx


def build_flat_matrix(sequences):
    Xf, bounds = [], []; ptr = 0
    for seq, lbl, rid in sequences:
        X = seq.numpy(); Xf.append(X)
        bounds.append((ptr, ptr+len(X))); ptr += len(X)
    return np.vstack(Xf), bounds


def repack(X_red, bounds, orig_seqs):
    return [(torch.tensor(X_red[s:e].astype(np.float32)), lbl, rid)
            for (s,e),(_, lbl, rid) in zip(bounds, orig_seqs)]


def fit_domain_pca(train_seqs, val_seqs, test_seqs, feat_cols,
                   n_per_domain=N_PER_DOMAIN, stream_tag=''):
    domain_idx = get_domain_indices(feat_cols)
    print(f'[{stream_tag}] Domains: ' + ', '.join(f'{d}({len(v)})' for d,v in domain_idx.items()))
    Xtr, b_tr = build_flat_matrix(train_seqs)
    Xva, b_va = build_flat_matrix(val_seqs)
    Xte, b_te = build_flat_matrix(test_seqs)
    parts_tr=[];parts_va=[];parts_te=[]; pca_reducers={}
    for domain, idxs in domain_idx.items():
        n_comp = min(n_per_domain, len(idxs), Xtr.shape[0]-1)
        pca = PCA(n_components=n_comp, random_state=42)
        parts_tr.append(pca.fit_transform(Xtr[:, idxs]))
        parts_va.append(pca.transform(Xva[:, idxs]))
        parts_te.append(pca.transform(Xte[:, idxs]))
        pca_reducers[domain] = {'pca': pca, 'indices': idxs, 'n_comp': n_comp}
    Xtr_r = np.hstack(parts_tr).astype('float32')
    Xva_r = np.hstack(parts_va).astype('float32')
    Xte_r = np.hstack(parts_te).astype('float32')
    n_out = Xtr_r.shape[1]
    print(f'  {Xtr.shape[1]}D → {n_out}D')
    return (repack(Xtr_r, b_tr, train_seqs),
            repack(Xva_r, b_va, val_seqs),
            repack(Xte_r, b_te, test_seqs),
            pca_reducers, n_out)


print('=== Domain PCA — FULL ===')
seq_tr_f_dp, seq_va_f_dp, seq_te_f_dp, pca_full, N_IN_FULL = fit_domain_pca(
    seq_tr_f, seq_va_f, seq_te_f, feat_full, stream_tag='FULL')

print('\n=== Domain PCA — ALG ===')
seq_tr_a_dp, seq_va_a_dp, seq_te_a_dp, pca_alg, N_IN_ALG = fit_domain_pca(
    seq_tr_a, seq_va_a, seq_te_a, feat_alg, stream_tag='ALG')


=== Domain PCA — FULL ===
[FULL] Domains: cognitive(73), functional(37), biomarker(65), demographics(14), medical_history(74), other(58)
  321D → 36D

=== Domain PCA — ALG ===
[ALG] Domains: cognitive(73), functional(37), biomarker(65), demographics(14), medical_history(74), other(28)
  291D → 36D


## 9. RNN Architecture & Training Utilities

In [10]:
class ADNIDataset(Dataset):
    def __init__(self, seqs): self.seqs = seqs
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        X, y, rid = self.seqs[i]
        return X, torch.tensor(y, dtype=torch.long), rid

def collate_fn(batch):
    seqs, labels, rids = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    padded  = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    return padded, torch.stack(labels), lengths, list(rids)

def make_loaders(tr, va, te, bs=32):
    kw = dict(collate_fn=collate_fn)
    return (DataLoader(ADNIDataset(tr), batch_size=bs, shuffle=True,  **kw),
            DataLoader(ADNIDataset(va), batch_size=bs, shuffle=False, **kw),
            DataLoader(ADNIDataset(te), batch_size=bs, shuffle=False, **kw))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


class TemporalAttention(nn.Module):
    def __init__(self, h):
        super().__init__(); self.attn = nn.Linear(h, 1, bias=False)
    def forward(self, hs, lengths):
        B, T, H = hs.shape
        scores = self.attn(hs).squeeze(-1)
        mask   = torch.arange(T, device=hs.device).unsqueeze(0) < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, float('-inf'))
        w      = F.softmax(scores, dim=1).unsqueeze(-1)
        return (w * hs).sum(1), w.squeeze(-1)


class ResidualRNNBlock(nn.Module):
    def __init__(self, cell, inp, hid, drop=0.0):
        super().__init__()
        self.rnn  = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}[cell](
            inp, hid, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(drop)
        self.norm = nn.LayerNorm(hid)
        self.proj = nn.Linear(inp, hid) if inp != hid else nn.Identity()
    def forward(self, x, hx=None):
        out, hx_new = self.rnn(x, hx)
        return self.norm(self.drop(out) + self.proj(x)), hx_new


class RecurrentClassifier(nn.Module):
    """
    RNN cfg2: cell_type='rnn', hidden_size=128, num_layers=3, dropout=0.3
    Stacked residual RNN blocks + temporal attention.
    Best from dim-red experiments: F1=0.8157 (full), 0.8187 (algerian).
    """
    def __init__(self, input_size, num_classes,
                 cell_type='rnn', hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.blocks     = nn.ModuleList([
            ResidualRNNBlock(cell_type, hidden_size, hidden_size, drop=dropout)
            for _ in range(num_layers)])
        self.attention  = TemporalAttention(hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_size // 2, num_classes))
    def forward(self, x, lengths):
        h = self.input_proj(x)
        for block in self.blocks: h, _ = block(h)
        context, _ = self.attention(h, lengths)
        return self.classifier(context)


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)


def compute_class_weights(seqs, nc):
    labels = np.array([s[1] for s in seqs])
    counts = np.bincount(labels, minlength=nc).astype(float)
    w = 1.0 / (counts + 1e-6)
    return torch.tensor(w / w.sum() * nc, dtype=torch.float32)


@torch.no_grad()
def evaluate_rnn(model, loader, crit):
    model.eval()
    all_p, all_l, all_pr = [], [], []
    for X, y, lengths, _ in loader:
        X, y, lengths = X.to(DEVICE), y.to(DEVICE), lengths.to(DEVICE)
        logits = model(X, lengths)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(y.cpu().numpy())
        all_pr.extend(F.softmax(logits, dim=1).cpu().numpy())
    return np.array(all_p), np.array(all_l), np.array(all_pr)


def train_rnn(model, tr_loader, va_loader, lr, epochs, patience, cw):
    crit  = FocalLoss(gamma=2.0, weight=cw.to(DEVICE))
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, steps_per_epoch=len(tr_loader),
        epochs=epochs, pct_start=0.15)
    best_f1, best_state, cnt = 0., None, 0
    for epoch in range(epochs):
        model.train()
        for X, y, lengths, _ in tr_loader:
            X, y, lengths = X.to(DEVICE), y.to(DEVICE), lengths.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(X, lengths), y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        preds, labels, _ = evaluate_rnn(model, va_loader, crit)
        vf1 = f1_score(labels, preds, average='macro', zero_division=0)
        if vf1 > best_f1: best_f1, best_state, cnt = vf1, copy.deepcopy(model.state_dict()), 0
        else: cnt += 1
        if cnt >= patience:
            print(f'  Early stop @ ep {epoch+1} | best val F1={best_f1:.4f}')
            break
    model.load_state_dict(best_state)
    return model, best_f1

print('Architecture and training utilities ready.')


Architecture and training utilities ready.


## 10. Train RNN cfg2 + Domain PCA (Both Streams)

In [11]:
cfg = RNN_CFG2   # hidden=128, layers=3, dropout=0.3, lr=3e-4, batch=16
trained_rnn = {}; rnn_results = []


def train_rnn_stream(stream_tag, tr_dp, va_dp, te_dp, n_in, le):
    print(f'\n{"="*60}')
    print(f'RNN cfg2 + domain PCA [{stream_tag.upper()}]  input={n_in}D')
    print(f'  Config: {cfg}')
    tr, va, te = make_loaders(tr_dp, va_dp, te_dp, bs=cfg['batch_size'])
    cw = compute_class_weights(tr_dp, len(le.classes_))
    model = RecurrentClassifier(
        n_in, len(le.classes_), cell_type='rnn',
        hidden_size=cfg['hidden_size'], num_layers=cfg['num_layers'], dropout=cfg['dropout']
    ).to(DEVICE)
    print(f'  Parameters: {count_params(model):,}')
    model, best_f1 = train_rnn(model, tr, va, lr=cfg['lr'],
                                epochs=RNN_EPOCHS, patience=RNN_PATIENCE, cw=cw)

    # Score on test
    te_loader = DataLoader(ADNIDataset(te_dp), batch_size=cfg['batch_size'],
                           shuffle=False, collate_fn=collate_fn)
    preds, labels, probs = evaluate_rnn(model, te_loader, FocalLoss())
    f1_mac = f1_score(labels, preds, average='macro', zero_division=0)
    acc    = (preds == labels).mean()
    try:    auc = roc_auc_score(np.eye(len(le.classes_))[labels], probs, multi_class='ovr', average='macro')
    except: auc = float('nan')
    print(f'  [{stream_tag}] Test: Acc={acc:.4f}  F1={f1_mac:.4f}  AUC={auc:.4f}')
    print(classification_report(labels, preds, target_names=le.classes_, zero_division=0))
    trained_rnn[stream_tag] = model
    rnn_results.append({'stream': stream_tag, 'model': 'RNN_cfg2_domain_pca',
                         'f1_macro': f1_mac, 'accuracy': acc, 'auc': auc})
    return model, best_f1


model_rnn_full, f1_rnn_full = train_rnn_stream('full', seq_tr_f_dp, seq_va_f_dp, seq_te_f_dp, N_IN_FULL, le_full)
model_rnn_alg,  f1_rnn_alg  = train_rnn_stream('alg',  seq_tr_a_dp, seq_va_a_dp, seq_te_a_dp, N_IN_ALG,  le_alg)



RNN cfg2 + domain PCA [FULL]  input=36D
  Config: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0003, 'batch_size': 16}
  Parameters: 113,155
  Early stop @ ep 27 | best val F1=0.7355
  [full] Test: Acc=0.7490  F1=0.7277  AUC=0.9407
                precision    recall  f1-score   support

            CN       0.99      0.71      0.83       144
MCI_converting       0.75      0.79      0.77        52
    MCI_stable       0.46      0.83      0.59        47

      accuracy                           0.75       243
     macro avg       0.73      0.78      0.73       243
  weighted avg       0.84      0.75      0.77       243


RNN cfg2 + domain PCA [ALG]  input=36D
  Config: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0003, 'batch_size': 16}
  Parameters: 113,155
  Early stop @ ep 25 | best val F1=0.7480
  [alg] Test: Acc=0.7819  F1=0.7508  AUC=0.9374
                precision    recall  f1-score   support

            CN       0.97      0.78      0.86     

## 11. Results Summary

In [12]:
summary_rows = []
for r in rnn_results:
    summary_rows.append({'stream': r['stream'], 'model': r['model'],
                          'f1_macro': r['f1_macro'], 'accuracy': r['accuracy'], 'auc': r['auc']})
for tag, res in [('full', lgb_full), ('alg', lgb_alg)]:
    summary_rows.append({'stream': tag, 'model': 'LightGBM_trajectory_summary',
                          'f1_macro': res['f1_macro'], 'accuracy': float('nan'), 'auc': res['auc']})

df_res = pd.DataFrame(summary_rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
print(df_res.round(4).to_string(index=False))


stream                       model  f1_macro  accuracy    auc
   alg LightGBM_trajectory_summary    0.7922       NaN 0.9587
  full LightGBM_trajectory_summary    0.7922       NaN 0.9587
   alg         RNN_cfg2_domain_pca    0.7508    0.7819 0.9374
  full         RNN_cfg2_domain_pca    0.7277    0.7490 0.9407


## 12. Save All Artifacts

In [13]:
def save_stream(stream_tag, rnn_model, lgb_res, pca_reducers,
                feat_cols, static_cols, le_target, cat_le_map, global_means, meta):
    out = f'{SAVE_ROOT}/{stream_tag}'
    os.makedirs(out, exist_ok=True)

    # ── RNN ─────────────────────────────────────────────────────────────
    torch.save(rnn_model.state_dict(), f'{out}/rnn_cfg2_domain_pca.pt')
    joblib.dump({'input_size': sum(v['n_comp'] for v in pca_reducers.values()),
                 'num_classes': len(le_target.classes_),
                 'cell_type': 'rnn', **RNN_CFG2}, f'{out}/rnn_config.pkl')
    joblib.dump(pca_reducers,  f'{out}/pca_reducers.pkl')
    joblib.dump({'mu': meta['mu'], 'std': meta['std']}, f'{out}/rnn_norm_stats.pkl')

    # ── LightGBM ─────────────────────────────────────────────────────────
    joblib.dump(lgb_res['model'],       f'{out}/lightgbm.pkl')
    joblib.dump(lgb_res['best_params'], f'{out}/lgb_best_params.pkl')
    joblib.dump(lgb_res['feat_cols'],   f'{out}/lgb_feature_cols.pkl')

    # ── Shared preprocessing ──────────────────────────────────────────────
    joblib.dump(feat_cols,    f'{out}/feature_cols.pkl')
    joblib.dump(static_cols,  f'{out}/static_cols.pkl')
    joblib.dump(le_target,    f'{out}/label_encoder.pkl')
    joblib.dump(cat_le_map,   f'{out}/cat_encoders.pkl')
    joblib.dump(global_means, f'{out}/global_means.pkl')

    print(f'  [{stream_tag}] All artifacts saved → {out}/')


save_stream('full', model_rnn_full, lgb_full, pca_full,
            feat_full, static_full, le_full, cat_le_full, gmeans_full, meta_f)
save_stream('alg',  model_rnn_alg,  lgb_alg,  pca_alg,
            feat_alg,  static_alg,  le_alg,  cat_le_alg,  gmeans_alg,  meta_a)
print('\nAll artifacts saved!')


  [full] All artifacts saved → ./stability_models/full/
  [alg] All artifacts saved → ./stability_models/alg/

All artifacts saved!


## 13. Loading Utilities

Run these in a fresh inference session.

In [14]:
def load_artifacts(stream_tag, save_root='./stability_models'):
    """Load all saved artifacts for one stream. Returns a single dict."""
    out = f'{save_root}/{stream_tag}'
    arts = {}
    # Shared preprocessing
    arts['feature_cols']  = joblib.load(f'{out}/feature_cols.pkl')
    arts['static_cols']   = joblib.load(f'{out}/static_cols.pkl')
    arts['label_encoder'] = joblib.load(f'{out}/label_encoder.pkl')
    arts['cat_encoders']  = joblib.load(f'{out}/cat_encoders.pkl')
    arts['global_means']  = joblib.load(f'{out}/global_means.pkl')
    # RNN-specific
    arts['pca_reducers']  = joblib.load(f'{out}/pca_reducers.pkl')
    arts['rnn_norm_stats']= joblib.load(f'{out}/rnn_norm_stats.pkl')
    arts['rnn_config']    = joblib.load(f'{out}/rnn_config.pkl')
    # LightGBM-specific
    arts['lightgbm']      = joblib.load(f'{out}/lightgbm.pkl')
    arts['lgb_feat_cols'] = joblib.load(f'{out}/lgb_feature_cols.pkl')
    return arts


def load_rnn_model(arts, device_inf=None):
    """Rebuild and load the RNN model from saved config + state dict."""
    if device_inf is None:
        device_inf = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cfg = arts['rnn_config']
    model = RecurrentClassifier(
        input_size=cfg['input_size'], num_classes=cfg['num_classes'],
        cell_type=cfg['cell_type'], hidden_size=cfg['hidden_size'],
        num_layers=cfg['num_layers'], dropout=cfg['dropout']).to(device_inf)
    # state dict path must be reconstructed — store stream_tag alongside arts in practice
    return model   # caller loads state dict: model.load_state_dict(torch.load(...))

print('Loading utilities defined.')


Loading utilities defined.


## 14. RNN Inference Functions

Input: chronologically ordered list of raw visit dicts.

In [15]:
def preprocess_visits_for_rnn(visit_list, arts):
    """
    Convert a list of raw visit dicts into a PCA-reduced (1, T, D) tensor
    ready for the RNN model.

    visit_list: [{'MMSCORE': 24, 'age': 72, 'FAQ': 3, ...}, ...]
                One dict per visit, ordered chronologically (earliest first).
    arts:       output of load_artifacts()

    Steps (mirrors the training pipeline exactly):
      1. Encode categorical columns with saved LabelEncoders
      2. Fill missing values with global_means
      3. Normalise with training-set mu/std
      4. Apply by-domain PCA with saved PCA reducers
    """
    feat_cols   = arts['feature_cols']
    cat_enc     = arts['cat_encoders']
    global_means= arts['global_means']
    mu          = arts['rnn_norm_stats']['mu']
    std         = arts['rnn_norm_stats']['std']
    pca_red     = arts['pca_reducers']

    rows = []
    prev_vals = {}   # for delta / rate features
    prev_month = None

    for v_idx, visit in enumerate(visit_list):
        row = []
        for col in feat_cols:
            if col in cat_enc:
                val = str(visit.get(col, 'UNKNOWN'))
                le  = cat_enc[col]
                row.append(float(le.transform([val])[0]) if val in le.classes_
                           else float(le.transform([le.classes_[0]])[0]))

            elif col.endswith('_missing'):
                base = col[:-len('_missing')]
                row.append(1.0 if (base not in visit or pd.isna(visit.get(base))) else 0.0)

            elif col.endswith('_delta'):
                base = col[:-len('_delta')]
                cur  = float(visit.get(base, np.nan))
                prv  = float(prev_vals.get(base, cur))
                row.append(cur - prv if not np.isnan(cur) and not np.isnan(prv) else 0.0)

            elif col.endswith('_rate'):
                base = col[:-len('_rate')]
                cur  = float(visit.get(base, np.nan))
                prv  = float(prev_vals.get(base, cur))
                gap  = max(float(visit.get('_month', v_idx)) - float(prev_month or 0), 1.0)
                row.append((cur - prv) / gap if not np.isnan(cur) and not np.isnan(prv) else 0.0)

            elif col.endswith('_from_bl'):
                base = col[:-len('_from_bl')]
                cur  = float(visit.get(base, np.nan))
                bl   = float(visit_list[0].get(base, np.nan))
                row.append((cur - bl) if not np.isnan(cur) and not np.isnan(bl) else 0.0)

            else:
                val = visit.get(col, np.nan)
                if pd.isna(val): val = float(global_means.get(col, 0.0))
                row.append(float(val))

        # Update prev state for delta/rate
        for c in TRAJECTORY_COLS:
            if c in visit: prev_vals[c] = float(visit[c])
        prev_month = visit.get('_month', v_idx)
        rows.append(row)

    X_raw  = np.array(rows, dtype=np.float32)
    X_norm = np.nan_to_num((X_raw - mu) / std, nan=0.0)

    # Apply domain PCA
    parts = []
    for domain, red in pca_red.items():
        idxs = [i for i in red['indices'] if i < X_norm.shape[1]]
        if not idxs: continue
        parts.append(red['pca'].transform(X_norm[:, idxs]))

    X_pca = np.hstack(parts).astype(np.float32)
    return torch.tensor(X_pca).unsqueeze(0), torch.tensor([len(visit_list)], dtype=torch.long)


def predict_rnn(visit_list, arts, model, device_inf=None):
    """
    Predict MCI stability for a patient from their visit history.

    visit_list : [{'MMSCORE': 26, 'MOCA': 25, 'FAQ': 2, ...}, ...]
                  One dict per visit, chronologically ordered.
    arts       : load_artifacts(stream_tag)
    model      : loaded RecurrentClassifier (load_rnn_model + load_state_dict)

    Returns: {prediction, class_index, probabilities, n_visits_used}
    """
    if device_inf is None: device_inf = next(model.parameters()).device
    model.eval()
    X, lengths = preprocess_visits_for_rnn(visit_list, arts)
    X = X.to(device_inf); lengths = lengths.to(device_inf)
    with torch.no_grad():
        probs = F.softmax(model(X, lengths), dim=1).cpu().numpy()[0]
    pred  = int(np.argmax(probs))
    le    = arts['label_encoder']
    return {'prediction':    le.inverse_transform([pred])[0],
            'class_index':   pred,
            'probabilities': {le.classes_[i]: float(p) for i, p in enumerate(probs)},
            'class_names':   list(le.classes_),
            'n_visits_used': len(visit_list)}

print('RNN inference functions ready.')


RNN inference functions ready.


## 15. LightGBM Inference Functions

Input: list of visit dicts (same format as RNN). The function internally computes the trajectory summary row before calling the model.

In [16]:
def compute_trajectory_summary_single(visit_list, arts):
    """
    Convert a list of visit dicts for ONE patient into a single trajectory
    summary feature row — the exact format LightGBM was trained on.

    visit_list: [{'MMSCORE': 26, 'MOCA': 25, ...}, ...]  ordered chronologically.
                Must include '_month' key (integer month number) if available.
    arts:       load_artifacts(stream_tag)

    Steps:
      1. Build a temporary single-patient DataFrame
      2. Encode categoricals with saved LabelEncoders
      3. Fill missing with global_means
      4. Compute per-column: slope, AUC, first/last/min/max/std, cum_decline
      5. Compute mean of delta/rate/from_bl derived columns
      6. Add static features (taken from first visit)

    Returns: numpy array (1, n_lgb_features)
    """
    cat_enc      = arts['cat_encoders']
    global_means = arts['global_means']
    feat_cols    = arts['feature_cols']
    lgb_feat     = arts['lgb_feat_cols']

    # ── Step 1: Build visit-level DataFrame ──────────────────────────────
    rows = []
    for v_idx, visit in enumerate(visit_list):
        row = {'_month': visit.get('_month', v_idx)}
        for col in feat_cols:
            if col.endswith(('_delta','_rate','_from_bl','_missing')): continue
            if col in cat_enc:
                val = str(visit.get(col, 'UNKNOWN'))
                le  = cat_enc[col]
                row[col] = float(le.transform([val])[0]) if val in le.classes_ \
                           else float(le.transform([le.classes_[0]])[0])
            else:
                val = visit.get(col, np.nan)
                row[col] = float(global_means.get(col, 0.0)) if pd.isna(val) else float(val)
        rows.append(row)
    g = pd.DataFrame(rows).sort_values('_month')
    months = g['_month'].values.astype(float)

    # ── Step 2: Build trajectory summary ─────────────────────────────────
    feat_row = {}
    feat_row['n_visits']          = len(g)
    feat_row['total_span_months'] = months[-1] - months[0] if len(months) > 1 else 0
    feat_row['mean_visit_gap']    = feat_row['total_span_months'] / max(len(g)-1, 1)

    traj_present = [c for c in TRAJECTORY_COLS if c in g.columns]
    for c in traj_present:
        vals = g[c].values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() < 1:
            for sfx in ('_slope','_auc','_first','_last','_min','_max','_std','_cum_decline'):
                feat_row[f'{c}{sfx}'] = 0.0
            continue
        v = vals[mask]; m = months[mask]
        slope = linregress(m, v).slope if len(v) >= 2 else 0.0
        span  = m[-1] - m[0]
        feat_row[f'{c}_slope']       = slope
        feat_row[f'{c}_auc']         = _trapz(v, m) / max(span, 1)
        feat_row[f'{c}_first']       = v[0]
        feat_row[f'{c}_last']        = v[-1]
        feat_row[f'{c}_min']         = v.min()
        feat_row[f'{c}_max']         = v.max()
        feat_row[f'{c}_std']         = v.std() if len(v) > 1 else 0.0
        diffs = np.diff(v)
        feat_row[f'{c}_cum_decline'] = diffs[diffs < 0].sum() if len(diffs) > 0 else 0.0

    # Delta / rate / from_bl means (computed over the visit window)
    prev = {}; prev_m = None
    delta_acc = {}; rate_acc = {}; bl_acc = {}
    first_visit = rows[0]
    for v_idx, visit in enumerate(visit_list):
        m_cur = visit.get('_month', v_idx)
        gap   = max(float(m_cur) - float(prev_m or 0), 1.0)
        for c in traj_present:
            cur = visit.get(c, np.nan)
            prv = prev.get(c, cur)
            bl  = first_visit.get(c, cur)
            if pd.isna(cur): cur = float(global_means.get(c, 0.0))
            if pd.isna(prv): prv = cur
            if pd.isna(bl):  bl  = cur
            delta_acc.setdefault(c, []).append(float(cur) - float(prv))
            rate_acc.setdefault(c, []).append((float(cur) - float(prv)) / gap)
            bl_acc.setdefault(c, []).append(float(cur) - float(bl))
            prev[c] = cur
        prev_m = m_cur

    for c in traj_present:
        feat_row[f'{c}_delta_mean'] = np.mean(delta_acc.get(c, [0.]))
        feat_row[f'{c}_delta_std']  = np.std(delta_acc.get(c, [0.]))
        feat_row[f'{c}_rate_mean']  = np.mean(rate_acc.get(c, [0.]))
        feat_row[f'{c}_rate_std']   = np.std(rate_acc.get(c, [0.]))
        feat_row[f'{c}_from_bl_mean'] = np.mean(bl_acc.get(c, [0.]))
        feat_row[f'{c}_from_bl_std']  = np.std(bl_acc.get(c, [0.]))

    for c in STATIC_FEATURES:
        if c in g.columns: feat_row[c] = g[c].iloc[0]

    # ── Step 3: Align to LightGBM feature list ────────────────────────────
    X = np.array([[feat_row.get(f, 0.0) for f in lgb_feat]], dtype=np.float32)
    return X


def predict_lightgbm(visit_list, arts):
    """
    Predict MCI stability using LightGBM from a patient's visit history.

    visit_list : [{'MMSCORE': 26, 'MOCA': 25, 'Abeta42': 180.0, ...}, ...]
                  One dict per visit, chronologically ordered.
                  Include '_month' key (integer) for accurate trajectory computation.
    arts       : load_artifacts(stream_tag)

    Returns: {prediction, class_index, probabilities, n_visits_used}
    """
    X = compute_trajectory_summary_single(visit_list, arts)
    model = arts['lightgbm']
    probs = model.predict_proba(X)[0]
    pred  = int(np.argmax(probs))
    le    = arts['label_encoder']
    return {'prediction':    le.inverse_transform([pred])[0],
            'class_index':   pred,
            'probabilities': {le.classes_[i]: float(p) for i, p in enumerate(probs)},
            'class_names':   list(le.classes_),
            'n_visits_used': len(visit_list)}

print('LightGBM inference functions ready.')


LightGBM inference functions ready.


## 16. Demo Inference

Shows how to reload and use both models on a new patient.

In [17]:
# ── Load artifacts (run in a fresh session) ───────────────────────────
arts_alg = load_artifacts('alg')

# ── Load RNN model ────────────────────────────────────────────────────
import os
rnn_path = f'{SAVE_ROOT}/alg/rnn_cfg2_domain_pca.pt'
rnn_inf  = load_rnn_model(arts_alg)
rnn_inf.load_state_dict(torch.load(rnn_path, map_location='cpu'))
rnn_inf.eval()

# ── Demo patient: 3 visits with some plausible values ─────────────────
patient_visits = [
    {'_month': 0,  'MMSCORE': 27, 'MOCA': 26, 'FAQ': 2, 'Abeta42': 185.0,
     'Tau181': 210.0, 'pTau181': 18.0, 'Abeta_ratio': 0.062,
     'age': 70, 'PTGENDER': '1', 'PTEDUCAT': 16, 'FFLUENCY': 14},
    {'_month': 12, 'MMSCORE': 25, 'MOCA': 24, 'FAQ': 5, 'Abeta42': 175.0,
     'Tau181': 245.0, 'pTau181': 21.0, 'Abeta_ratio': 0.058,
     'age': 71, 'PTGENDER': '1', 'PTEDUCAT': 16, 'FFLUENCY': 12},
    {'_month': 24, 'MMSCORE': 23, 'MOCA': 21, 'FAQ': 9, 'Abeta42': 162.0,
     'Tau181': 290.0, 'pTau181': 26.0, 'Abeta_ratio': 0.051,
     'age': 72, 'PTGENDER': '1', 'PTEDUCAT': 16, 'FFLUENCY': 10},
]

# RNN prediction
result_rnn = predict_rnn(patient_visits, arts_alg, rnn_inf)
print('RNN prediction:')
for k, v in result_rnn.items(): print(f'  {k}: {v}')

print()

# LightGBM prediction (same visit list, different pipeline)
result_lgb = predict_lightgbm(patient_visits, arts_alg)
print('LightGBM prediction:')
for k, v in result_lgb.items(): print(f'  {k}: {v}')


RNN prediction:
  prediction: MCI_converting
  class_index: 1
  probabilities: {'CN': 0.1302548199892044, 'MCI_converting': 0.8367214798927307, 'MCI_stable': 0.03302371874451637}
  class_names: ['CN', 'MCI_converting', 'MCI_stable']
  n_visits_used: 3

LightGBM prediction:
  prediction: MCI_converting
  class_index: 1
  probabilities: {'CN': 0.34236046738869236, 'MCI_converting': 0.5665867309075849, 'MCI_stable': 0.09105280170372268}
  class_names: ['CN', 'MCI_converting', 'MCI_stable']
  n_visits_used: 3
